## PlantGuard‑Meta — Colab GPU Training (Qwen + GRPO + HF Space reward)

This notebook trains a **Qwen instruct model** using **TRL GRPO** where the reward comes from your **deployed OpenEnv environment on Hugging Face Spaces**.

### Why this is the “best fit” for your problem
- **Environment reward is the source of truth** (no offline heuristic).
- **GRPO** is a strong default for LLM RL post‑training.
- Uses **4‑bit loading + LoRA** so it fits typical Colab GPUs.

### Environment
- **HF Space URL**: `https://narendra78-plant-governor-env.hf.space`

### Outputs
- `outputs/training_log.csv`
- `outputs/reward_curve.png`

> Recommended model default for Colab: `Qwen/Qwen2.5-3B-Instruct` (4‑bit + LoRA).
> If you have A100 40GB, you can switch to `Qwen/Qwen2.5-7B-Instruct`.


In [ ]:
# --- Install (Colab) ---
!pip -q install -U \
  "trl>=0.25.0" \
  "transformers>=4.45" \
  "accelerate>=0.34" \
  "datasets>=2.20" \
  "bitsandbytes" \
  "torch" \
  "requests" \
  "pandas" "numpy" "matplotlib"

import torch, sys
print('python', sys.version.split()[0])
print('torch', torch.__version__)
print('cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))
    print('vram_gb', round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2))


In [ ]:
# --- Config ---
import os, json, time, random, csv
from typing import Any, Dict, List

import numpy as np
import pandas as pd
import requests

BASE_URL = "https://narendra78-plant-governor-env.hf.space"
RESET_URL = f"{BASE_URL}/reset"
STEP_URL  = f"{BASE_URL}/step"
HEALTH_URL = f"{BASE_URL}/health"

print('health', requests.get(HEALTH_URL, timeout=30).json())

# Best default for 16GB GPU (4-bit + LoRA friendly)
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
# If you hit OOM, fall back to:
# MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)
TRAIN_LOG = os.path.join(OUT_DIR, "training_log.csv")

# Training knobs
MAX_STEPS = 300          # increase to 1000+ for better learning
NUM_GENERATIONS = 2      # keep small if your Space has limited concurrency
N_PROMPTS = 256          # prompts built from env.reset seeds
SEED0 = 42

random.seed(SEED0)
np.random.seed(SEED0)


In [ ]:
# --- Environment HTTP helpers ---

def env_reset(seed: int) -> Dict[str, Any]:
    r = requests.post(RESET_URL, json={"seed": int(seed)}, timeout=60)
    r.raise_for_status()
    return r.json()


def env_step(action: Dict[str, Any]) -> Dict[str, Any]:
    r = requests.post(STEP_URL, json=action, timeout=60)
    r.raise_for_status()
    return r.json()


VALID_TOOLS = {
    "run_diagnostic",
    "adjust_load",
    "dispatch_repair",
    "order_spare_part",
    "do_nothing",
}


def obs_to_prompt(obs: Dict[str, Any], step: int = 0, last_action: str = "none") -> str:
    return f"""You are PlantGuard‑Meta, an AI plant operations manager.

Step {step}/720 | Sensors:
- air_temp: {obs.get('air_temp', 0):.1f} K
- process_temp: {obs.get('process_temp', 0):.1f} K
- rotational_speed: {obs.get('rotational_speed', 0):.0f} rpm
- torque: {obs.get('torque', 0):.1f} Nm
- tool_wear: {obs.get('tool_wear', 0):.0f} min
- shift_hour: {obs.get('shift_hour', 0)}
- remaining_budget: ${obs.get('remaining_budget', 0):.0f}
- last_action: {last_action}

Pick ONE tool and justify it.
Return ONLY valid JSON:
{{"tool":"<tool>","reasoning":"<why>","load_reduction":null}}

Valid tools: run_diagnostic, adjust_load, dispatch_repair, order_spare_part, do_nothing.
"""


def parse_action(text: str) -> Dict[str, Any]:
    t = (text or "").strip()
    if "```json" in t:
        t = t.split("```json", 1)[1].split("```", 1)[0].strip()
    elif "```" in t:
        t = t.split("```", 1)[1].split("```", 1)[0].strip()

    start = t.find("{")
    end = t.rfind("}") + 1
    if start != -1 and end > start:
        t = t[start:end]

    try:
        data = json.loads(t)
    except Exception:
        return {"tool": "do_nothing", "reasoning": "parse_error", "load_reduction": None}

    tool = data.get("tool", "do_nothing")
    reasoning = data.get("reasoning", "") or ""
    lr = data.get("load_reduction", None)

    if tool not in VALID_TOOLS:
        tool = "do_nothing"

    if tool == "adjust_load" and lr is not None:
        try:
            lr = float(lr)
            lr = max(0.1, min(0.9, lr))
        except Exception:
            lr = None
    else:
        lr = None

    return {"tool": tool, "reasoning": reasoning, "load_reduction": lr}


In [ ]:
# --- Load Qwen in 4-bit (GRPO) ---
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import GRPOConfig, GRPOTrainer


tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    load_in_4bit=True,
    device_map="auto",
)

print('Loaded model:', MODEL_ID)


In [ ]:
# --- Build training prompts by sampling real env resets ---
prompts: List[Dict[str, Any]] = []
for i in range(N_PROMPTS):
    seed = SEED0 + i
    payload = env_reset(seed)
    obs = payload.get("observation", payload)
    prompts.append({"prompt": obs_to_prompt(obs, step=0, last_action="none")})

train_ds = Dataset.from_list(prompts)
print('train_ds', len(train_ds))
print('\nExample prompt:\n', train_ds[0]['prompt'][:400])


In [ ]:
# --- Rollout: generate → step HF Space → use env reward ---
# We also log env rewards over time so the notebook produces judge-ready training curves.

ROLL_OUT_LOG: List[Dict[str, Any]] = []  # filled during training
_rollout_call_idx = 0


def rollout_func(prompts: List[str], args: GRPOConfig, processing_class):
    global _rollout_call_idx

    tok = processing_class
    completions = []
    env_rewards = []

    for p in prompts:
        inputs = tok(p, return_tensors="pt", truncation=True).to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=220,
                do_sample=True,
                temperature=1.0,
                top_p=0.95,
                pad_token_id=tok.eos_token_id,
            )

        completion_text = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        action = parse_action(completion_text)
        step_payload = env_step(action)
        r = float(step_payload.get("reward") or 0.0)

        completions.append([{"content": completion_text}])
        env_rewards.append(r)

        # Log one row per env interaction (rollout call, per-prompt).
        ROLL_OUT_LOG.append(
            {
                "rollout_call": int(_rollout_call_idx),
                "reward": float(r),
                "tool": action.get("tool", ""),
                "done": bool(step_payload.get("done", False)),
            }
        )

    _rollout_call_idx += 1
    return {"completions": completions, "env_reward": env_rewards}


def reward_from_env(prompts, completions, env_reward=None, **kwargs):
    if env_reward is None:
        return [0.0 for _ in completions]
    return [float(x) for x in env_reward]


In [ ]:
# --- Train (GRPO) ---
# This is the minimal “correct” loop: env reward is computed by the Space.

args = GRPOConfig(
    output_dir=os.path.join(OUT_DIR, "grpo_out"),
    max_steps=MAX_STEPS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    num_generations=NUM_GENERATIONS,
    max_completion_length=220,
    logging_steps=5,
    save_steps=50,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=reward_from_env,
    train_dataset=train_ds,
    args=args,
    rollout_func=lambda prompts, args, processing_class: rollout_func(prompts, args, processing_class),
)

trainer.train()
print('Saved to', args.output_dir)


In [ ]:
# --- Write judge-ready training logs + plots (from real env rewards) ---
import matplotlib.pyplot as plt

log_df = pd.DataFrame(ROLL_OUT_LOG)
if len(log_df) == 0:
    raise RuntimeError("No rewards logged. Run the training cell first.")

# Aggregate by rollout_call (a proxy for training progression)
agg = log_df.groupby("rollout_call")["reward"].agg(["mean", "std", "count"]).reset_index()
agg = agg.rename(columns={"rollout_call": "step", "mean": "reward_mean", "std": "reward_std", "count": "n"})

agg.to_csv(TRAIN_LOG, index=False)
print('Wrote', TRAIN_LOG)

# Reward curve
plt.figure(figsize=(8, 4))
plt.plot(agg["step"], agg["reward_mean"], label="mean reward")
plt.fill_between(
    agg["step"],
    agg["reward_mean"] - agg["reward_std"].fillna(0),
    agg["reward_mean"] + agg["reward_std"].fillna(0),
    alpha=0.2,
    label="±1 std",
)
plt.xlabel("training step (rollout call)")
plt.ylabel("env reward")
plt.title("PlantGuard‑Meta GRPO training (env reward)")
plt.legend()
plt.tight_layout()
reward_png = os.path.join(OUT_DIR, "reward_curve.png")
plt.savefig(reward_png, dpi=180)
plt.show()

print('Saved', reward_png)


In [ ]:
# --- Optional: quick reward plot from a manual eval rollouts (recommended) ---
import matplotlib.pyplot as plt


def eval_random_policy(n_episodes: int = 10, max_steps: int = 50):
    rewards = []
    for i in range(n_episodes):
        env_reset(SEED0 + 10_000 + i)
        total = 0.0
        for t in range(max_steps):
            tool = random.choice(list(VALID_TOOLS))
            action = {"tool": tool, "reasoning": "eval baseline", "load_reduction": None}
            payload = env_step(action)
            total += float(payload.get('reward') or 0.0)
            if payload.get('done', False):
                break
        rewards.append(total)
    return rewards

baseline_rewards = eval_random_policy(10, 50)
plt.figure(figsize=(6,4))
plt.bar(list(range(len(baseline_rewards))), baseline_rewards)
plt.xlabel('episode')
plt.ylabel('total reward (50 steps)')
plt.title('Random baseline (short horizon)')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'baseline_comparison.png'), dpi=180)
plt.show()

print('Saved', os.path.join(OUT_DIR, 'baseline_comparison.png'))
